#  OBJECTIVE

**This notebook focuses on packaging and validating the trained Random Forest model for production deployment** — serialising it with all dependencies into a portable `.pkl` pipeline, verifying it with simulated real-world inputs, building an optimised prediction function, writing edge-case unit tests, and documenting the modular architecture for clean Streamlit integration.

> **Input:** `final_random_forest_model.pkl` | **Output:** `flight_price_prediction_pipeline.pkl` | **Purpose:** Production-ready prediction API

---
##  Step: Model Serialisation — Deployment Package Creation

**Why:** A trained model object in memory is session-bound — it disappears when the Colab runtime ends. `joblib.dump()` serialises the model, feature list, name, and version into a single portable dictionary (the "deployment package"). This atomic structure ensures the Streamlit backend always loads a *consistent, self-describing* model — it knows exactly which features to expect and what version is running, preventing silent prediction errors from feature-order mismatches.

**Why:** The `deployment_package` pattern (rather than saving just the model) is a production best practice — it bundles all information the backend needs to validate inputs, make predictions, and log model version for monitoring and audit trails.

task 1:Convert the final model into a serialized .pkl file, ensuring it includes all dependencies for loading.

In [ ]:
import joblib
import numpy as np
import pandas as pd

In [ ]:
BASE_PATH = "/content/drive/MyDrive/AirFair-Vista"

rf_model = joblib.load(
    f"{BASE_PATH}/models/final_random_forest_model.pkl"
)

In [ ]:
features = joblib.load(
    f"{BASE_PATH}/models/rf_12_model_features.pkl"
)

In [ ]:
#create a deployment pipeline object
deployment_package = {

"model": rf_model,

"features": features,

"model_name": "RandomForest_FlightPrice",

"version": "1.0"

}

In [ ]:
#serialize final deployment model
joblib.dump(

deployment_package,

f"{BASE_PATH}/models/flight_price_prediction_pipeline.pkl"

)

['/content/drive/MyDrive/AirFair-Vista/models/flight_price_prediction_pipeline.pkl']

In [ ]:
#verify the model loads coreectly
loaded_pipeline = joblib.load(

f"{BASE_PATH}/models/flight_price_prediction_pipeline.pkl"

)

print(loaded_pipeline.keys())

dict_keys(['model', 'features', 'model_name', 'version'])


---
##  Step: Model Loading Test with Simulated Real-World Input

**Why:** Serialisation success (no error on `joblib.dump`) does not guarantee correct prediction behaviour on new inputs. This test simulates a real booking request — creating a sample input dictionary with realistic flight attributes, converting it to the exact feature format the model expects, and verifying the predicted price is in a plausible range (₹2,000–₹80,000). This "smoke test" catches serialisation corruption, feature-order bugs, and inverse-transform errors before the model reaches production.

**Why:** `reindex(columns=features, fill_value=0)` is a critical safety step — it ensures input columns are always in the training order, regardless of how the frontend passes data. Missing features default to 0 rather than causing a crash.

task 2: Test model loading in a separate script to verify compatibility with real-world input data.

In [ ]:
#code for test script
import joblib
import pandas as pd
import numpy as np

# Project path
BASE_PATH = "/content/drive/MyDrive/AirFair-Vista"

# Load serialized deployment pipeline
pipeline = joblib.load(
    f"{BASE_PATH}/models/flight_price_prediction_pipeline.pkl"
)

model = pipeline["model"]
features = pipeline["features"]

print("Model loaded successfully!")
print("Model name:", pipeline["model_name"])
print("Features expected:", len(features))

# Simulate real-world input data
sample_input = {

    "total_duration_mins": 180,
    "Airline_mean_price": 9000,
    "duration_hours": 3,
    "journey_month": 5,
    "journey_day": 15,
    "journey_weekday": 2,
    "Source_freq": 1200,
    "dep_hour": 10

}

# Convert to dataframe
input_df = pd.DataFrame([sample_input])

# Ensure feature order matches training
input_df = input_df.reindex(columns=features, fill_value=0)

# Make prediction
prediction = model.predict(input_df)

# Convert log price back to normal price
predicted_price = np.expm1(prediction)

print("Predicted Flight Price:", predicted_price[0])

Model loaded successfully!
Model name: RandomForest_FlightPrice
Features expected: 20
Predicted Flight Price: 5769.29876616167


---
##  Step: Optimised Prediction Function with Feature Engineering

**Why:** The raw prediction requires multiple steps: input validation → feature construction → column reordering → model inference → log-inverse transform. Encapsulating this into a single `predict_flight_price(input_data)` function creates a clean API contract. The Streamlit frontend calls one function and receives a price — it doesn't need to know about log-transformation or feature ordering. This separation of concerns is the core principle of production ML systems.

**Why:** Modular function design also enables independent testing of each preprocessing step (date extraction, stop encoding, frequency lookup) before integration — dramatically reducing debugging time when the Streamlit app is connected.

task 3: Write an optimized function to accept user inputs, apply feature engineering, and preprocess data before predictions.

In [ ]:
#prediction engine.py

---
##  Step: Unit Testing with Edge-Case Inputs

**Why:** Edge cases reveal brittle assumptions that unit tests on typical inputs miss. Testing: (1) minimum/maximum duration values, (2) unknown airline names, (3) missing optional fields, (4) zero-stop vs. 4-stop flights ensures the prediction function handles every input the user might provide in the Streamlit UI without crashing. This is especially important before public deployment — a price prediction app that crashes on unusual input destroys user trust immediately.

**Why:** Unit tests also serve as living documentation — they show future developers exactly which input variations the system was validated against, forming the test suite for model updates.

task 4: Conduct unit testing on the preprocessing and prediction functions with a variety of edge-case inputs.

In [ ]:
#in upper file last block

---
##  Step: Modular Script Refactoring for Streamlit Integration

**Why:** Jupyter notebooks run sequentially and have global state — they are unsuitable for production serving. Refactoring preprocessing logic into modular functions (`preprocess_input()`, `predict_price()`, `validate_input()`) enables them to be imported as Python modules by the Streamlit app. This architecture means model improvements only require updating `prediction_engine.py` — without touching the frontend code — following the separation of concerns principle critical for maintainable ML applications.

task 5: Refactor and document the preprocessing and model scripts, ensuring modularity for smooth integration with Streamlit.

---
##  Next Step → Notebook 14: Streamlit Frontend Development

The prediction pipeline is packaged, tested, and modular. **Notebook 14** builds the user-facing Streamlit web application — creating input widgets for all flight attributes (airline dropdown, route selection, departure time, number of stops), calling the backend prediction function, and displaying the estimated price — connecting the full ML pipeline to an interactive user interface for the first time.